In [6]:
# extended from hl_subsets_202417.ipynb
# now with F1 as a mean of F1 socores from several thermodynamic models
# 260730 ddd

# hl_subsets_202417.ipynb
#   revised from hl_subsets.ipynb
#   (running all again)
#
#   save each HL split in a separate CSV file 
#   sequences less than 512 nts
#   241217 fix 512 max length

In [7]:
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

# Data

In [10]:
DATA_PATH = "../../../data/bpRNA/"
 
dref = pd.read_csv(DATA_PATH + "bpRNA.csv")
dref.set_index("id", inplace=True)
display(dref.shape)
dref["len"] = dref["sequence"].str.len() 
MAX_L = 512 # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
upto = list(dref.loc[dref.len < MAX_L].index)  
dref = dref.loc[upto]

display(dref)

(18537, 2)

,sequence,base_pairs,len
id,,,
bpRNA_CRW_15639,UCCCUGGUGAAAUUAGCGCUGUGGAACCACCCGAUCCCAUCCCGAA...,"[[1, 118], [2, 117], [3, 116], [4, 115], [5, 1...",119
bpRNA_CRW_15847,UAGAGGUUUUUAUUAACUUGGUGGUUUUAGCAAAAAUGAAACACCC...,"[[15, 124], [16, 123], [17, 122], [18, 121], [...",125
bpRNA_CRW_15871,CCCGGUGGCUAUCGCGGAGGGGCCACACCCGUUCCCUUUCCGAAUA...,"[[1, 116], [2, 115], [3, 114], [4, 113], [5, 1...",121
bpRNA_CRW_15994,UUUGCGGUGGUCAUAGCCCUGGUGAGACGCCCGGUCCCAUUCCGAA...,"[[4, 117], [5, 116], [6, 115], [7, 114], [8, 1...",117
bpRNA_CRW_16021,AGCUUGACGAAUAUAGUUUGUGGGACCAACCUGAAACCAUUCCGAA...,"[[1, 120], [2, 119], [3, 118], [4, 117], [5, 1...",121
...,...,...,...
URS0000D72239_12908_1-146,GGCGCGUGCUUAAUGAGUACACGCUUUGUAAAAGAAAAUGUAGUUC...,"[[1, 25], [2, 24], [3, 23], [4, 22], [6, 20], ...",146
URS0000D7223A_12908_1-142,GGAUCGUAGCUAAUGACUACCAUCUUCGGGGUUAAGAUACCAGCAU...,"[[1, 25], [2, 24], [3, 23], [4, 22], [6, 20], ...",142
URS0000D7223B_12908_1-95,GGGUCGCAGUUAAUGACUACCGCCUCCAGUGUACAAUGGUCUCCAA...,"[[1, 25], [2, 24], [3, 23], [4, 22], [6, 20], ...",95


In [ ]:
CLASSICAL_SUMMARY_PATH = "../../../results/bpRNA/classical_summary.pkl"
classical_summary = pd.read_pickle(CLASSICAL_SUMMARY_PATH)

selected_methods = ['RNAfold', 'RNAstructure', 'LinearFoldV', 'LinearPartitionV']
classical_summary = classical_summary[classical_summary.method.isin(selected_methods)]
metrics_mean = classical_summary.groupby("id").f1.mean().to_frame()
display(metrics_mean)

# just for comparison
metrics_rnafold = classical_summary.loc[classical_summary.method == "RNAfold"].f1.to_frame()
metrics_rnafold

,f1
id,
AACY020454254.1_1909-1844,0.379310
AACY020454584.1_1474-1546,0.438259
AACY020454584.1_604-676,0.533422
AACY020496190.1_1503-1438,0.652778
AACY021048934.1_360-303,0.000000
...,...
bpRNA_tmRNA_8,0.596261
bpRNA_tmRNA_86,0.482649
bpRNA_tmRNA_94,0.459858


,f1
id,
bpRNA_RFAM_37268,0.424242
bpRNA_RFAM_12641,0.755556
bpRNA_RFAM_21314,0.901961
bpRNA_CRW_42535,0.565217
bpRNA_RFAM_11145,0.928571
...,...
CP007653.1_1278983-1278851,1.000000
URS0000D68500_12908_1-68,0.421053
URS0000D6B7C1_12908_1-41,0.736842


# Human-learning subsets

In [ ]:
def hl_split(metrics, base_file_name, verbose=False):
    splitS = [5+5*j for j in range(1, 19)]
    random.seed(42)
    all_splits = []

    if verbose:
        print(f"=== {base_file_name} ===")
    for i, sp in enumerate(splitS):
        fold_name = "hl" + str(sp)

        hl_test_sp = list(metrics.loc[metrics.f1 < sp/100.0].index)
        # to-do: check for empty test
        hl_train_sp = list(set(metrics.index)-set(hl_test_sp)) 

        # 10% for validation (simil k-fold)
        hl_valid_sp = random.sample(hl_train_sp, len(hl_train_sp)//10)
        hl_train_sp = list(set(hl_train_sp)-set(hl_valid_sp))
        
        if verbose:
            print(f" {sp:<3} tst:{len(hl_test_sp):5d}, trn:{len(hl_train_sp):5d}, val:{len(hl_valid_sp):5d}")
            
        for seq_id in hl_test_sp:
            all_splits.append([fold_name, i, "test", seq_id])
        for seq_id in hl_valid_sp:
            all_splits.append([fold_name, i, "valid", seq_id])
        for seq_id in hl_train_sp:
            all_splits.append([fold_name, i, "train", seq_id])


    all_splits_df = pd.DataFrame(all_splits, columns=["fold_name", "fold_number", "partition", "id"])
    # sort columns as in the other split CSVs
    all_splits_df = all_splits_df[["id", "fold_name", "fold_number", "partition"]]
    all_splits_df.to_csv(DATA_PATH + "create_splits/splits/hl-folds/hl-f1means/" + base_file_name + ".csv", index=False)
    all_splits_df 

In [ ]:
hl_split(metrics_mean, "hl_folds_split_F1mean")
hl_split(metrics_rnafold, "hl_folds_split_F1rnafold") # just for comparison

# Comparisons

In [ ]:
# verification of the new splits for RNAfold vs the old RNAfold splits

old_splits_df = pd.read_csv(DATA_PATH + "create_splits/splits/hl-folds/hl-f1means/" + "hl_folds_split_F1mean.csv", index_col="id")
new_splits_df = pd.read_csv(DATA_PATH + "create_splits/splits/hl-folds/hl-f1means/" + "hl_folds_split_F1rnafold.csv", index_col="id")

# joint train+val splits
old_train_val = old_splits_df[old_splits_df['partition'].isin(['train', 'valid'])]
new_train_val = new_splits_df[new_splits_df['partition'].isin(['train', 'valid'])]

old_test = old_splits_df[old_splits_df['partition'] == 'test']
new_test = new_splits_df[new_splits_df['partition'] == 'test']

for sp in old_splits_df.fold_name.unique():
    # compare the test ids
    old_test_ids = set(old_test[old_test.fold_name == sp].index)
    new_test_ids = set(new_test[new_test.fold_name == sp].index)
    if old_test_ids != new_test_ids:
        print(f"Test sets differ for fold {sp}:")
        print(f"Old test ids: {old_test_ids}")
        print(f"New test ids: {new_test_ids}")
    else:
        print(f"ts=", end="")

    # compare the train+valid ids
    old_train_val_ids = set(old_train_val[old_train_val.fold_name == sp].index)
    new_train_val_ids = set(new_train_val[new_train_val.fold_name == sp].index)
    if old_train_val_ids != new_train_val_ids:
        print(f"Train+Valid sets differ for fold {sp}:")
        print(f"Old train+valid ids: {old_train_val_ids}")
        print(f"New train+valid ids: {new_train_val_ids}")
    else:
        print(f"tv=", end="")

In [ ]:
# count differences between the new RNAfold splits and F1mean splits

F1m_splits_df = pd.read_csv(DATA_PATH + "create_splits/splits/hl-folds/hl-f1means/" + "hl_folds_split_F1mean.csv", index_col="id")
new_splits_df = pd.read_csv(DATA_PATH + "create_splits/splits/hl-folds/hl-f1means/" + "hl_folds_split_F1rnafold.csv", index_col="id")

# joint train+val splits
F1m_train_val = F1m_splits_df[F1m_splits_df['partition'].isin(['train', 'valid'])]
new_train_val = new_splits_df[new_splits_df['partition'].isin(['train', 'valid'])]

F1m_test = F1m_splits_df[F1m_splits_df['partition'] == 'test']
new_test = new_splits_df[new_splits_df['partition'] == 'test']

print(f"Number of different ids")
for sp in F1m_splits_df.fold_name.unique():
    # compare the test ids
    F1m_test_ids = set(F1m_test[F1m_test.fold_name == sp].index)
    new_test_ids = set(new_test[new_test.fold_name == sp].index)
    dif = len(F1m_test_ids - new_test_ids)
    N = len(F1m_test_ids)
    print(f"Fold {sp} --- tst: {dif:4d} of {N:4d} => {dif*100/N:5.2f}%", end="")

    # compare the train+valid ids
    F1m_train_val_ids = set(F1m_train_val[F1m_train_val.fold_name == sp].index)
    new_train_val_ids = set(new_train_val[new_train_val.fold_name == sp].index)
    dif = len(F1m_train_val_ids - new_train_val_ids)
    N = len(F1m_train_val_ids)
    print(f" --- t+v: {dif:4d} of {N:4d} => {dif*100/N:5.2f}%")
